In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Processed folder से क्लीन किए गए डेटासेट्स लोड करना
processed_dir = '../data/processed/'
df_orders = pd.read_csv(os.path.join(processed_dir, 'cleaned_orders.csv'))
df_customers = pd.read_csv(os.path.join(processed_dir, 'cleaned_customers.csv'))
df_restaurants = pd.read_csv(os.path.join(processed_dir, 'cleaned_restaurants.csv'))

print("Data loaded successfully for feature engineering!")

Data loaded successfully for feature engineering!


In [2]:
# उदाहरण के लिए: आर्डर टाइम से समय (Hour, Day, Month) के नए फीचर्स निकालना
if 'order_time' in df_orders.columns:
    df_orders['order_time'] = pd.to_datetime(df_orders['order_time'])
    df_orders['order_hour'] = df_orders['order_time'].dt.hour
    df_orders['order_day_of_week'] = df_orders['order_time'].dt.dayofweek
    df_orders['is_weekend'] = df_orders['order_day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# ग्राहक की कुल आर्डर फ्रीक्वेन्सी या अन्य मेट्रिक्स जोड़ना
if 'customer_id' in df_orders.columns:
    customer_freq = df_orders['customer_id'].value_counts().reset_index()
    customer_freq.columns = ['customer_id', 'order_frequency']
    df_orders = df_orders.merge(customer_freq, on='customer_id', how='left')

print("New features created successfully!")

New features created successfully!


In [3]:
# कैटेगोरिकल कॉलम्स के लिए एनकोडिंग
categorical_cols = df_orders.select_dtypes(include=['object']).columns
label_encoders = {}

for col in categorical_cols:
    if col != 'order_id':  # आईडी कॉलम्स को छोड़कर
        le = LabelEncoder()
        df_orders[col] = le.fit_transform(df_orders[col].astype(str))
        label_encoders[col] = le

# न्यूमेरिकल फीचर्स के लिए StandardScaler का उपयोग
scaler = StandardScaler()
numeric_cols = df_orders.select_dtypes(include=[np.number]).columns
# टार्गेट या आईडी को छोड़कर स्केलिंग करना
scale_features = [col for col in numeric_cols if col not in ['order_id', 'delivery_time', 'churn']]

if scale_features:
    df_orders[scale_features] = scaler.fit_transform(df_orders[scale_features])

print("Encoding and scaling completed!")

C:\Users\HP\AppData\Local\Temp\ipykernel_13108\3531085173.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_orders.select_dtypes(include=['object']).columns


Encoding and scaling completed!


In [4]:
# फीचर-इंजीनियर डेटा को प्रोसेसड फोल्डर में सेव करना
output_path = '../data/processed/featured_orders.csv'
df_orders.to_csv(output_path, index=False)
print(f"Featured dataset saved successfully to {output_path}")

Featured dataset saved successfully to ../data/processed/featured_orders.csv
